# 30 — Segmentation Training

Trains the segmentation models across the four input scenarios and two
architectures. Driven by `stages/train_segmentation.py :: main()`.

- **scenarios** (`exps`): `single_date`, `mt_base`, `gsi`, `rf`
- **architectures** (`archs`): `deeplabv3plus_cbam` (ResNet-50), `segformer` (MiT-B2)
- Train/val/test = spatial **block** split (no patch-adjacency leakage)
- Metrics + artifacts logged to MLflow; checkpoints under `ml_models/`

> GPU required. A full run (4 scenarios × 2 archs × up to 150 epochs) is long —
> start with a single short run below.

In [ ]:
# Make the pipeline importable as `crop_mapping_pipeline` regardless of the
# checkout directory name (this repo is `cropmap-remote-sensing-exps`; the
# GPU deploy dir is `crop_mapping_pipeline`). Also silence MLflow telemetry.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))
print('S2 bands/date:', C.S2_BAND_NAMES)
print('S2 train dir :', C.S2_TRAIN_DIR)
print('CDL train    :', C.CDL_TRAIN)

In [ ]:
from crop_mapping_pipeline.stages import train_segmentation as T
print('MLflow  :', C.MLFLOW_TRACKING_URI)
print('Batch   :', C.BATCH_SIZE, '| max epochs', C.MAX_EPOCHS, '| patch', C.PATCH_SIZE)
print('Split   : block', C.BLOCK_SIZE, 'px  |  seed', C.SEED)
for a, cfg in C.ARCH_CFG.items(): print(f'  {a}: {cfg}')

## 1. Single short run (smoke test)

GSI scenario on SegFormer, few epochs. Verifies the data → split → train → eval path.

In [ ]:
T.main(
    exps=['gsi'],
    archs=['segformer'],
    top_k=[C.SELECT_TOP_K_PER_CROP],
    loss='wce',
    epochs=3,          # smoke test; remove to use config MAX_EPOCHS
    skip_ndvi=True,
)

## 2. Full experiment matrix

All four scenarios × both architectures at the config epoch budget.
Uncomment to launch (long).

In [ ]:
# T.main(
#     exps=['single_date', 'mt_base', 'gsi', 'rf'],
#     archs=['deeplabv3plus_cbam', 'segformer'],
#     top_k=[C.SELECT_TOP_K_PER_CROP],
#     loss='wce',
# )

## 3. Seed-grid stability

Re-runs the matrix per seed (re-seeds the spatial split; tags runs `_seed{N}`,
logs `seed` param). Use the CLI for the full grid:

```bash
python stages/train_segmentation.py --exp gsi --arch segformer \
       --top-k 20 --seed-grid 42 123 456 789
```

Or per-seed from Python:

In [ ]:
# for s in [42, 123, 456]:
#     T.main(exps=['gsi'], archs=['segformer'], top_k=[C.SELECT_TOP_K_PER_CROP], seed=s)
#
# Then aggregate mean/std across seeds:
# from crop_mapping_pipeline.stages import collect_seed_grid_metrics